In [14]:
import socket

# UDP_IP = "udp2dmx2"  # IP-Adresse deines ESP32
UDP_IP = "192.168.178.55"  # IP-Adresse deines ESP32
UDP_PORT = 6454

def send_dmx_command(command: str):
    sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
    sock.sendto(command.encode(), (UDP_IP, UDP_PORT))
    print(f"Gesendet: {command}")

# Beispiele für Kommandos:

# # Direktwert 129 auf Kanal 2
# send_dmx_command("DMXP5#0#2")
# send_dmx_command("DMXC9#150#2")
# send_dmx_command("DMXP9#0")

# Prozentwert 55% auf Kanal 3, Geschwindigkeit 1
# send_dmx_command("DMXP4#55#1")

# # Prozentwert 56% auf Kanal 3, Geschwindigkeit 2
# send_dmx_command("DMXP3#56#2#2")

# # RGB-Wert: R=12, G=66, B=3 -> R + G*1000 + B*1000000
rgb_value = 80 + 5*1000 + 20*1000000  # = 3066012
send_dmx_command(f"DMXR9#{rgb_value}#2")
# send_dmx_command(f"DMXR10#{rgb_value}#2")
# send_dmx_command(f"DMXR10#{rgb_value}#1#1")

# # Tunable White (Typ V): 70% auf Kanal 5 (WW@5, CW@6)
# send_dmx_command("DMXV5#70")

# # Tunable White (Typ W): 70% auf Kanal 7 (CW@7, WW@8)

send_dmx_command("DMXW7#000128#3")
# send_dmx_command("DMXL1#20250030#3")

# # Du kannst auch eine Liste von Kommandos senden:
# commands = [
#     "DMXC1#255",   # Kanal 1 auf 255 setzen
#     "DMXP2#50",    # Kanal 2 auf 50%
#     "DMXR10#3066012",  # RGB auf Kanal 10
# ]


# for cmd in commands:
#     send_dmx_command(cmd)


Gesendet: DMXR9#20005080#2
Gesendet: DMXW7#000128#3


In [10]:
send_dmx_command("DMXL9#200802000#2") 


Gesendet: DMXL9#200802000#2


In [6]:
#Rest API Python Beispiel
import requests
import json

ESP32_IP = "udp2dmx2"  # <– hier deine ESP32-IP eintragen

# # Einzelwert ändern (PATCH):
# patch_data = {
#     "ct_config": {
#         "13": 2400,
#         "14": 6500
#     }
# }
# response = requests.post(f"http://{ESP32_IP}/config/patch", json=patch_data)
# print("PATCH Einzelwert:", response.status_code, response.text)

# # MinMax-Werte ändern (PATCH):
# patch_minmax = {
#     "default_ct": {
#         "min": 3000
#     }
# }
# response = requests.post(f"http://{ESP32_IP}/config/patch", json=patch_minmax)
# print("PATCH MinMax:", response.status_code, response.text)


# #Beispiel längere Json Datei
# patch_minmax = {
#     "ct_config": {
#         "1": 2500,
#         "7": 6000,
#         "8": 2000
#     },
#     "default_ct": {
#         "min": 3400,
#         "max": 6600
#     }
# }
patch_minmax = {
"hostname": "udp2dmx"
    }


response = requests.post(f"http://{ESP32_IP}/config/patch", json=patch_minmax)
print("PATCH MinMax:", response.status_code, response.text)



# # Ganze Datei lesen (GET):
# response = requests.get(f"http://{ESP32_IP}/config")
# print("GET Config:", response.status_code)
# print(response.json())

# # Ganze Datei ersetzen (POST):
# with open("main\components\spiffs_image\spiffs\config.json", "r") as f:
#     config_data = json.load(f)

# response = requests.post(f"http://{ESP32_IP}/config", json=config_data)
# print("POST Full Config:", response.status_code, response.text)


PATCH MinMax: 200 OK


In [11]:
#Beispielaufrufe für die REEST API:

# Einzelwert ändern:
curl -X POST http://<ESP32-IP>/config/patch \
     -H "Content-Type: application/json" \
     -d '{"ct_config": {"10": 5100}}'

# MinMax Werte ändern
curl -X POST http://<ESP32-IP>/config/patch \
     -H "Content-Type: application/json" \
     -d '{"default_ct": {"min": 3000}}'

# Ganze Datei lesen:
curl http://<ESP32-IP>/config

#Ganze Datei ersetzen/schreiben:
curl -X POST http://<ESP32-IP>/config \
     -H "Content-Type: application/json" \
     -d @ct_config.json

SyntaxError: invalid syntax (2666489373.py, line 4)

In [15]:
# === Mode-Tests (Kanäle 8–12) ===
# Unterstützte Modi im Firmware-Parser:
#   DMXC (0..255), DMXP (0..100%), DMXR (RGB, 3 Kanäle), DMXW (Tunable White, 2 Kanäle), DMXL (CT, 2 Kanäle)

import time

CHANNELS_1CH = list(range(8, 13))          # 8..12
CHANNELS_2CH = list(range(8, 12))          # 8..11 (braucht 2 Kanäle)
CHANNELS_3CH = list(range(8, 11))          # 8..10 (braucht 3 Kanäle)

# Loxone-speed: 255=sofort, sonst wird im ESP umgerechnet (siehe Logs für fade=ms)
SPEED_DIM = 210   # sichtbar dimmen (ca. ~720ms)
SPEED_STEP = 255  # sofort

def _send(cmd: str, pause_s: float = 0.25):
    send_dmx_command(cmd)
    time.sleep(pause_s)

def _rgb_value(r: int, g: int, b: int) -> int:
    # Firmware: r = value%1000, g=(value/1000)%1000, b=(value/1_000_000)%1000
    return int(r) + int(g) * 1000 + int(b) * 1_000_000

def _tw_value(ww: int, cw: int) -> int:
    # Firmware: ww=(value/1000)%1000, cw=value%1000  => value=ww*1000 + cw
    return int(ww) * 1000 + int(cw)

def _ct_value(brightness_percent: int, color_temp_k: int) -> int:
    # Firmware (DMXL): value 200000000..209999999
    # brightness = (value/10000)%1000 (0..100), ct=value%10000 (Kelvin)
    return 200_000_000 + int(brightness_percent) * 10_000 + int(color_temp_k)

print("== DMX Mode Test Start ==")

# 1) DMXC: Direktwerte (0..255) auf Kanäle 8..12
print("-- DMXC: Direktwerte --")
for ch in CHANNELS_1CH:
    for v in (0, 32, 128, 255, 0):
        _send(f"DMXC{ch}#{v}#{SPEED_STEP}")

# 2) DMXP: Prozent-Dimmen (0..100%) auf Kanäle 8..12
print("-- DMXP: Dimm-Sweep --")
for ch in CHANNELS_1CH:
    for pct in (0, 10, 25, 50, 75, 100, 75, 50, 25, 10, 0):
        _send(f"DMXP{ch}#{pct}#{SPEED_DIM}", pause_s=0.35)

# 3) DMXR: RGB Tests (Kanäle 8-10, 9-11, 10-12)
print("-- DMXR: RGB Farben --")
rgb_tests = [
    (255, 0, 0),
    (0, 255, 0),
    (0, 0, 255),
    (255, 255, 255),
    (0, 0, 0),
    (80, 5, 20),
    (0, 0, 0),
 ]
for start_ch in CHANNELS_3CH:
    for (r, g, b) in rgb_tests:
        _send(f"DMXR{start_ch}#{_rgb_value(r, g, b)}#{SPEED_DIM}", pause_s=0.4)

# 4) DMXW: Tunable White (WW/CW) auf Kanalpaaren 8..11
print("-- DMXW: Tunable White (WW/CW) --")
tw_tests = [
    (0, 0),
    (255, 0),
    (0, 255),
    (128, 128),
    (0, 0),
 ]
for start_ch in CHANNELS_2CH:
    for (ww, cw) in tw_tests:
        _send(f"DMXW{start_ch}#{_tw_value(ww, cw)}#{SPEED_DIM}", pause_s=0.4)

# 5) DMXL: CT (Helligkeit% + Kelvin) auf Kanalpaaren 8..11
print("-- DMXL: CT (Brightness% + Kelvin) --")
ct_tests = [
    (0, 2700),
    (25, 2700),
    (50, 4000),
    (75, 6500),
    (100, 6500),
    (0, 2700),
 ]
for start_ch in CHANNELS_2CH:
    for (bri, ct) in ct_tests:
        _send(f"DMXL{start_ch}#{_ct_value(bri, ct)}#{SPEED_DIM}", pause_s=0.45)

print("== DMX Mode Test Done ==")

== DMX Mode Test Start ==
-- DMXC: Direktwerte --
Gesendet: DMXC8#0#255
Gesendet: DMXC8#32#255
Gesendet: DMXC8#128#255
Gesendet: DMXC8#255#255
Gesendet: DMXC8#0#255
Gesendet: DMXC9#0#255
Gesendet: DMXC9#32#255
Gesendet: DMXC9#128#255
Gesendet: DMXC9#255#255
Gesendet: DMXC9#0#255
Gesendet: DMXC10#0#255
Gesendet: DMXC10#32#255
Gesendet: DMXC10#128#255
Gesendet: DMXC10#255#255
Gesendet: DMXC10#0#255
Gesendet: DMXC11#0#255
Gesendet: DMXC11#32#255
Gesendet: DMXC11#128#255
Gesendet: DMXC11#255#255
Gesendet: DMXC11#0#255
Gesendet: DMXC12#0#255
Gesendet: DMXC12#32#255
Gesendet: DMXC12#128#255
Gesendet: DMXC12#255#255
Gesendet: DMXC12#0#255
-- DMXP: Dimm-Sweep --
Gesendet: DMXP8#0#210
Gesendet: DMXP8#10#210
Gesendet: DMXP8#25#210
Gesendet: DMXP8#50#210
Gesendet: DMXP8#75#210
Gesendet: DMXP8#100#210
Gesendet: DMXP8#75#210
Gesendet: DMXP8#50#210
Gesendet: DMXP8#25#210
Gesendet: DMXP8#10#210
Gesendet: DMXP8#0#210
Gesendet: DMXP9#0#210
Gesendet: DMXP9#10#210
Gesendet: DMXP9#25#210
Gesendet: DMXP9#5